# AATD Project — predicting product discounts

## Data understanding (EDA)

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from scipy import stats

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (AdaBoostClassifier, BaggingClassifier,
                              GradientBoostingClassifier, RandomForestClassifier)
from sklearn.impute import KNNImputer, MissingIndicator, SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, confusion_matrix,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import (GridSearchCV, StratifiedKFold, cross_val_score, cross_validate,
                                     train_test_split)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import OneHotEncoder, PowerTransformer, RobustScaler, StandardScaler
from sklearn.tree import DecisionTreeClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler

warnings.filterwarnings("ignore")
from matplotlib.colors import to_rgb

from matplotlib.colors import LinearSegmentedColormap

# Colour palette: just blue and orange (light tones when more than two series are needed)
BLUE, ORANGE = "#1F77B4", "#FF7F0E"
LIGHT_BLUE, LIGHT_ORANGE = "#9ECAE1", "#FDBE85"
C_NO, C_YES = BLUE, ORANGE                       # class y = 0 (blue) and y = 1 (orange)
C_DIST, C_BOX, C_IMP = BLUE, LIGHT_BLUE, BLUE    # distributions, boxplots, importance
METRIC_COLORS = [BLUE, ORANGE, LIGHT_BLUE, LIGHT_ORANGE]   # accuracy, recall, precision, F-measure
CLASS_PALETTE = {0: C_NO, 1: C_YES}
CMAP_BO = LinearSegmentedColormap.from_list("blue_orange", [BLUE, "#FFFFFF", ORANGE])   # diverging, for correlations
sns.set_theme(style="whitegrid", palette=[BLUE, ORANGE])

# ------------------------------ configuration ------------------------------
SEED = 42
PREP_SEED = 43         # the preparation experiments use different folds from the final model assessment
DATA_DIR = Path(".")   # training.csv and the test file must be in this folder (default: next to the notebook)
TRAIN_FILE = DATA_DIR / "training.csv"
# the test file is called test.csv or task.csv, depending on where it came from
TEST_FILE = next((p for p in (DATA_DIR / "test.csv", DATA_DIR / "task.csv") if p.exists()), DATA_DIR / "test.csv")
FIG_DIR, OUT_DIR = Path("figures"), Path("outputs")
FIG_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

N_SPLITS = 5          # outer cross-validation folds (model assessment)
INNER_SPLITS = 3      # inner folds used to tune hyper-parameters
TUNE = True           # False -> default hyper-parameters (much faster)
MAX_CAT_LEVELS = 20   # discrete variables with more levels are treated as numeric
RARE_CAT_FREQ = 0.01  # categories rarer than this are grouped into "infrequent"
CORR_THRESHOLD = 0.90 # |Pearson r| above which a numeric variable is redundant (see 2.3)
DROP_DUPLICATES = False  # True -> remove exact duplicate rows (features + y) from the training set
MIN_GAIN = 0.003      # a preparation option replaces the first (simplest) one only if it improves the mean accuracy by this much
N_JOBS = -1


def savefig(name):
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")


TABLE_DIR = Path("tables")
TABLE_DIR.mkdir(exist_ok=True)


def style_for_display(out, color=None, highlight_row=None, decimals=4):
    """Coloured table with the same number of decimals as the saved files."""
    sty = _style_core(out, color, highlight_row)
    try:
        return sty.format(precision=decimals)
    except Exception:
        return sty


def _style_core(out, color=None, highlight_row=None):
    """Coloured version of a table, for the notebook only (the saved files have no styling)."""
    try:
        header = {"selector": "th", "props": [("background-color", "#1F4E79"), ("color", "white"),
                                               ("text-align", "center")]}
        zebra = {"selector": "tbody tr:nth-child(even) td", "props": [("background-color", "#EEF3F8")]}
        cols = list(out.select_dtypes("number").columns)
        if highlight_row is not None:
            return out.style.set_table_styles([header]).apply(
                lambda r: ["background-color: #FDD0A2; font-weight: bold" if r.name == highlight_row else ""
                           for _ in r], axis=1)
        if color == "best" and cols:        # best (highest) value of each column in orange
            return out.style.set_table_styles([header]).highlight_max(subset=cols, color="#FDD0A2")
        if color == "gradient" and cols:    # colour scale from white to orange
            return out.style.set_table_styles([header]).background_gradient(cmap="Oranges", subset=cols)
        return out.style.set_table_styles([header, zebra])
    except Exception:
        return out


def show_table(df, name, caption, index=True, decimals=4, color=None, highlight_row=None):
    """Shows a table with its caption and saves it as tables/<name>.csv and .md, ready for the report."""
    if isinstance(df, pd.Series):
        df = df.to_frame()
    out = df.round(decimals)
    if isinstance(out.columns, pd.MultiIndex):          # "accuracy / RandomForest" instead of a tuple
        out.columns = [" / ".join(map(str, c)) for c in out.columns]
    if isinstance(out.index, pd.MultiIndex):            # index levels become ordinary columns
        out, index = out.reset_index(), False
    print(f"Table: {caption}")
    display(style_for_display(out, color, highlight_row, decimals))
    out.to_csv(TABLE_DIR / f"{name}.csv", index=index)
    try:
        (TABLE_DIR / f"{name}.md").write_text(out.to_markdown(index=index))
    except Exception:
        pass  # the markdown export needs `tabulate`; the CSV is always written


def make_grid(n, ncols=3, w=4.2, h=3.2):
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(w * ncols, h * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes[n:]:
        ax.set_visible(False)
    return fig, axes

### Load data and basic inspection

In [ ]:
train = pd.read_csv(TRAIN_FILE)
print("Shape:", train.shape)
display(train.head())
train.info()

# The identifier must be unique; it is removed from the modeling data
assert train["i"].is_unique, "identifier 'i' is not unique"
train = train.drop(columns="i")

d_vars = [c for c in train.columns if c.startswith("d")]
x_vars = [c for c in train.columns if c.startswith("x")]
print(f"\n{len(d_vars)} discrete variables: {d_vars}")
print(f"{len(x_vars)} continuous variables: {x_vars}")

y = train["y"].astype(int)
X = train.drop(columns="y")

# discrete variables with few levels -> categorical (one-hot); many levels -> numeric
cat_cols = [c for c in d_vars if X[c].nunique() <= MAX_CAT_LEVELS]
num_cols = x_vars + [c for c in d_vars if c not in cat_cols]
print("\nTreated as categorical:", cat_cols)
print("Treated as numeric    :", num_cols)

### Descriptive analysis

In [ ]:
# --- continuous variables
desc_x = train[x_vars].describe().T
desc_x["skew"] = train[x_vars].skew()
desc_x["kurtosis"] = train[x_vars].kurt()
desc_x["n_unique"] = train[x_vars].nunique()
desc_x["missing_%"] = 100 * train[x_vars].isna().mean()
show_table(desc_x, "desc_continuous", "Descriptive statistics of the continuous variables", decimals=3)

# --- discrete variables
desc_d = pd.DataFrame({
    "n_unique": train[d_vars].nunique(),
    "missing_%": 100 * train[d_vars].isna().mean(),
    "mode": train[d_vars].mode().iloc[0],
    "mode_freq_%": [100 * train[c].value_counts(normalize=True).iloc[0] for c in d_vars],
    "min_cat_freq_%": [100 * train[c].value_counts(normalize=True).iloc[-1] for c in d_vars],
})
show_table(desc_d, "desc_discrete", "Descriptive statistics of the discrete variables", decimals=2)

rare = {c: vc[vc < RARE_CAT_FREQ].index.tolist()
        for c in d_vars if (vc := train[c].value_counts(normalize=True)).lt(RARE_CAT_FREQ).any()}
print(f"Categories with frequency < {RARE_CAT_FREQ:.0%}:", rare or "none")

### Structure of the discrete variables

In [ ]:
binary_d = [c for c in d_vars if set(train[c].dropna().unique()) <= {0, 1}]
other_d = [c for c in d_vars if c not in binary_d]
print(f"Binary (0/1) discrete variables [{len(binary_d)}]: {binary_d}")
print(f"Other discrete variables        [{len(other_d)}]: {other_d}")

if binary_d:
    prevalence = (100 * train[binary_d].mean()).sort_values().rename("pct_rows_with_1")
    show_table(prevalence, "binary_d_prevalence", "Share of rows (%) with each 0/1 indicator equal to 1", decimals=2)

    n_active = train[binary_d].sum(axis=1)
    show_table(n_active.value_counts().sort_index().rename_axis("active_indicators").rename("n_rows"),
               "binary_d_active_per_row", "Number of active 0/1 indicators per row")
    if (n_active == 1).all():
        print("-> exactly one indicator is active in every row: the variables are dummies of ONE categorical variable.")
    elif (n_active <= 1).all():
        print("-> at most one indicator is active per row: probably dummies of one categorical variable "
              "(rows with none = reference category).")

### Target distribution

In [ ]:
counts = y.value_counts().sort_index()
props = y.value_counts(normalize=True).sort_index()
show_table(pd.DataFrame({"count": counts, "proportion": props}), "target_counts", "Distribution of the target y")
print(f"Majority-class baseline accuracy: {props.max():.4f}")
print(f"Imbalance ratio (majority/minority): {counts.max() / counts.min():.2f}")

fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(counts.index.astype(str), counts.values, color=[C_NO, C_YES])
ax.set(xlabel="y (1 = discount)", ylabel="count", title="Target distribution")
savefig("target_distribution")
plt.show()

### Histograms

In [ ]:
# Continuous variables: overall distribution
fig, axes = make_grid(len(x_vars))
for ax, v in zip(axes, x_vars):
    sns.histplot(train[v].dropna(), kde=True, ax=ax, color=C_DIST)
    ax.set_title(v)
fig.tight_layout()
savefig("hist_x")
plt.show()

# Continuous variables: distribution by class
fig, axes = make_grid(len(x_vars))
for ax, v in zip(axes, x_vars):
    sns.histplot(data=train, x=v, hue="y", element="step", stat="density", common_norm=False, ax=ax,
                 palette=CLASS_PALETTE)
    ax.set_title(f"{v} by y")
fig.tight_layout()
savefig("hist_x_by_y")
plt.show()

In [ ]:
# Discrete variables: frequency and share of y=1 per level
fig, axes = make_grid(len(d_vars))
for ax, v in zip(axes, d_vars):
    if train[v].nunique() <= MAX_CAT_LEVELS:
        ct = pd.crosstab(train[v], train["y"])
        ct.plot(kind="bar", stacked=True, ax=ax, width=0.85, legend=(v == d_vars[0]), color=[C_NO, C_YES])
    else:
        sns.histplot(train[v].dropna(), ax=ax, color=C_DIST)
    ax.set_title(v)
fig.tight_layout()
savefig("hist_d")
plt.show()

levels = pd.concat({v: train.groupby(v)["y"].agg(n_rows="size", pct_y1="mean") for v in d_vars},
                   names=["variable", "level"])
levels["pct_y1"] = 100 * levels["pct_y1"]
show_table(levels, "d_levels_vs_y", "Rows and share of y=1 (%) per level of each discrete variable", decimals=1)

### Boxplots

In [ ]:
# Raw boxplots (original scale)
fig, axes = make_grid(len(x_vars))
for ax, v in zip(axes, x_vars):
    sns.boxplot(y=train[v], ax=ax, color=C_BOX)
    ax.set_title(v)
fig.tight_layout()
savefig("box_x")
plt.show()

# All variables on a comparable (z-score) scale
Z = (train[x_vars] - train[x_vars].mean()) / train[x_vars].std()
fig, ax = plt.subplots(figsize=(1.0 * len(x_vars) + 3, 4))
sns.boxplot(data=Z, ax=ax, color=C_BOX)
ax.set_title("Continuous variables (z-scores)")
savefig("box_x_standardised")
plt.show()

# By class
fig, axes = make_grid(len(x_vars))
for ax, v in zip(axes, x_vars):
    sns.boxplot(data=train, x="y", y=v, hue="y", palette=CLASS_PALETTE, legend=False, ax=ax)
    ax.set_title(f"{v} by y")
fig.tight_layout()
savefig("box_x_by_y")
plt.show()

### Correlations

In [ ]:
def high_corr_pairs(df, thr):
    corr = df.corr()
    cols = corr.columns
    pairs = [(cols[i], cols[j], corr.iloc[i, j])
             for i in range(len(cols)) for j in range(i + 1, len(cols))
             if abs(corr.iloc[i, j]) >= thr]
    out = pd.DataFrame(pairs, columns=["var_1", "var_2", "r"])
    return out.sort_values("r", key=lambda s: s.abs(), ascending=False)


fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, method in zip(axes, ["pearson", "spearman"]):
    corr = train[x_vars + ["y"]].corr(method=method)
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap=CMAP_BO, center=0,
                vmin=-1, vmax=1, ax=ax, annot_kws={"size": 7})
    ax.set_title(f"{method.capitalize()} correlation (x variables and y)")
fig.tight_layout()
savefig("correlations_x")
plt.show()

show_table(high_corr_pairs(train[x_vars], 0.7), "corr_pairs_x",
           "Pairs of continuous variables with absolute Pearson correlation >= 0.7", index=False, decimals=3)

corr_y = train[x_vars + ["y"]].corr()["y"].drop("y").sort_values(key=abs, ascending=False)
show_table(corr_y.to_frame("r_with_y"), "corr_with_y", "Pearson correlation of each continuous variable with y", decimals=3,
           color="gradient")

In [ ]:
# Discrete variables: association with y and between each other (Cramér's V)
def cramers_v(a, b):
    ct = pd.crosstab(a, b)
    if min(ct.shape) < 2:
        return np.nan
    chi2 = stats.chi2_contingency(ct, correction=False)[0]
    return np.sqrt((chi2 / ct.values.sum()) / (min(ct.shape) - 1))


assoc = pd.DataFrame({
    "cramers_v_with_y": [cramers_v(train[c], train["y"]) for c in d_vars],
    "chi2_p_value": [stats.chi2_contingency(pd.crosstab(train[c], train["y"]))[1] for c in d_vars],
}, index=d_vars)
show_table(assoc.sort_values("cramers_v_with_y", ascending=False), "assoc_d_y",
           "Association of each discrete variable with y (Cramér's V and chi-square p-value)")

if len(cat_cols) > 1:
    V = pd.DataFrame([[cramers_v(train[a], train[b]) for b in cat_cols] for a in cat_cols],
                     index=cat_cols, columns=cat_cols)
    fig, ax = plt.subplots(figsize=(0.7 * len(cat_cols) + 3, 0.6 * len(cat_cols) + 2.5))
    sns.heatmap(V, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=ax,
                annot_kws={"size": 7})
    ax.set_title("Association between discrete variables (Cramér's V)")
    savefig("cramers_v_d")
    plt.show()

### Data quality overview: missing values and outliers

In [ ]:
missing = pd.DataFrame({"n_missing": train.isna().sum(), "pct_missing": (100 * train.isna().mean()).round(2)})
missing = missing[missing["n_missing"] > 0]
if len(missing):
    show_table(missing, "missing_values", "Variables with missing values", decimals=2)
    # Is missingness related to the target?
    miss_vs_y = pd.concat({v: train.groupby(train[v].isna())["y"].agg(n_rows="size", pct_y1="mean")
                           .rename(index={True: "missing", False: "observed"}) for v in missing.index},
                          names=["variable", "value"])
    miss_vs_y["pct_y1"] = 100 * miss_vs_y["pct_y1"]
    show_table(miss_vs_y, "missing_vs_y", "Share of y=1 (%) when the variable is missing vs. observed", decimals=2)
else:
    print("No missing values in the training data.")


def tukey_outliers(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())


outliers = pd.DataFrame({"n_outliers": [tukey_outliers(train[v]) for v in x_vars]}, index=x_vars)
outliers["pct_outliers"] = (100 * outliers["n_outliers"] / len(train)).round(2)
outliers["skew"] = train[x_vars].skew().round(2)
outliers["pct_zeros"] = (100 * (train[x_vars] == 0).mean()).round(2)
show_table(outliers.sort_values("n_outliers", ascending=False), "outliers_tukey",
           "Outliers per continuous variable (Tukey 1.5 x IQR rule)", decimals=2)

## Data preparation

In [ ]:
# ---------------------------- custom transformers ----------------------------
class TukeyClipper(BaseEstimator, TransformerMixin):
    """Clips each column to [Q1 - k*IQR, Q3 + k*IQR] computed on the training data (winsorising)."""

    def __init__(self, k=1.5):
        self.k = k

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        q1, q3 = np.nanpercentile(X, 25, axis=0), np.nanpercentile(X, 75, axis=0)
        iqr = q3 - q1
        self.lower_, self.upper_ = q1 - self.k * iqr, q3 + self.k * iqr
        return self

    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lower_, self.upper_)


class CorrelationFilter(BaseEstimator, TransformerMixin):
    """Drops a column when |Pearson r| with an already-kept column exceeds `threshold`."""

    def __init__(self, threshold=0.9):
        self.threshold = threshold

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        corr = np.abs(np.atleast_2d(np.corrcoef(X, rowvar=False)))
        keep = []
        for j in range(corr.shape[0]):
            if not any(corr[i, j] > self.threshold for i in keep):
                keep.append(j)
        self.keep_ = np.array(keep)
        return self

    def transform(self, X):
        return np.asarray(X, dtype=float)[:, self.keep_]


def make_preprocessor(missing="median", outliers="keep", corr_threshold=None, use_d=True):
    imputer = {"median": SimpleImputer(strategy="median"),
               "mean": SimpleImputer(strategy="mean"),
               "knn": KNNImputer(n_neighbors=5)}[missing]
    steps = [("imputer", imputer)]
    if outliers == "clip":
        steps += [("clip", TukeyClipper()), ("scale", StandardScaler())]
    elif outliers == "power":
        steps += [("power", PowerTransformer(method="yeo-johnson"))]
    elif outliers == "robust":
        steps += [("scale", RobustScaler())]
    elif outliers == "keep":
        steps += [("scale", StandardScaler())]
    else:
        raise ValueError(outliers)
    if corr_threshold is not None:
        steps.append(("corr", CorrelationFilter(corr_threshold)))

    blocks = [
        ("num", SkPipeline(steps), num_cols),
        # flags the rows whose values were imputed (only for columns that had NaN in training)
        ("miss_flag", MissingIndicator(features="missing-only", error_on_new=False), x_vars),
    ]
    if cat_cols and use_d:
        blocks.append(("cat", SkPipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            # drop="if_binary": a 0/1 variable stays ONE column (no redundant twin column)
            ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", drop="if_binary",
                                     min_frequency=RARE_CAT_FREQ, sparse_output=False)),
        ]), cat_cols))
    return ColumnTransformer(blocks, remainder="drop")


def build_pipeline(model, missing="median", outliers="keep", balancing="none", corr_threshold=None, use_d=True):
    steps = [("prep", make_preprocessor(missing, outliers, corr_threshold, use_d))]
    if balancing == "smote":
        steps.append(("sampler", SMOTE(random_state=SEED)))
    elif balancing == "undersample":
        steps.append(("sampler", RandomUnderSampler(random_state=SEED)))
    elif balancing != "none":
        raise ValueError(balancing)
    steps.append(("clf", clone(model)))
    return ImbPipeline(steps)

### Duplicates

In [ ]:
REF_MODELS = {"kNN": KNeighborsClassifier(n_neighbors=11),
              "RandomForest": RandomForestClassifier(n_estimators=200, random_state=SEED)}

feature_cols = list(X.columns)
same_features = X.duplicated(keep=False)                 # features also appear in another row (any label)
n_surplus = int(train.duplicated().sum())                # rows that repeat an earlier row exactly (features + y)
print(f"Rows whose features appear in at least one other row: {same_features.sum()} ({100 * same_features.mean():.2f}%)")
print(f"Exact duplicates (features + y), surplus rows       : {n_surplus} ({100 * n_surplus / len(train):.2f}%)")

# Identical features, different label -> conflicting labels
grp = train.groupby(feature_cols, dropna=False)["y"]
g_sum, g_size = grp.transform("sum"), grp.transform("size")
conflict = (g_sum > 0) & (g_sum < g_size)
print(f"Rows in groups with conflicting labels              : {int(conflict.sum())} ({100 * conflict.mean():.2f}%)")
ceiling = (np.maximum(g_sum, g_size - g_sum) / g_size).sum() / len(train)   # majority label counted once per group
print(f"Accuracy ceiling implied by identical-feature groups: {ceiling:.4f} "
      "(best any model could do on these data, counting only exact feature matches)")

if TEST_FILE.exists():
    _test = pd.read_csv(TEST_FILE)
    seen = _test[feature_cols].merge(X.drop_duplicates(), on=feature_cols, how="left", indicator=True)["_merge"].eq("both")
    print(f"Test rows whose features also occur in the training set: {int(seen.sum())} of {len(_test)} ({100 * seen.mean():.2f}%)")

dup_rows = [("Rows whose features appear in another row", int(same_features.sum()), len(train)),
            ("Exact duplicates (surplus rows)", n_surplus, len(train)),
            ("Rows in groups with conflicting labels", int(conflict.sum()), len(train))]
if TEST_FILE.exists():
    dup_rows.append(("Test rows whose features occur in the training set", int(seen.sum()), len(_test)))
dup_summary = pd.DataFrame(dup_rows, columns=["measure", "n_rows", "out_of"]).set_index("measure")
dup_summary["pct"] = 100 * dup_summary["n_rows"] / dup_summary["out_of"]
show_table(dup_summary, "duplicates_summary", "Duplicate rows in the training set and overlap with the test set", decimals=2)

if n_surplus:
    # How much do duplicates inflate the cross-validated accuracy? Same folds recipe, with and without them.
    keep_first = (~train.duplicated()).values
    cv_dup = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=PREP_SEED)
    rows = []
    for label, mask in {"with duplicates": np.ones(len(train), dtype=bool),
                        "exact duplicates removed": keep_first}.items():
        for name, model in REF_MODELS.items():
            res = cross_validate(build_pipeline(model), X[mask], y[mask], cv=cv_dup,
                                 scoring=["accuracy", "f1"], n_jobs=N_JOBS)
            rows.append({"data": label, "n_rows": int(mask.sum()), "model": name,
                         "accuracy": res["test_accuracy"].mean(), "f1": res["test_f1"].mean()})
    show_table(pd.DataFrame(rows).pivot(index=["data", "n_rows"], columns="model", values=["accuracy", "f1"]),
               "duplicates_impact", "Cross-validated accuracy and F-measure with and without exact duplicates")

if DROP_DUPLICATES and n_surplus:
    train = train.drop_duplicates().reset_index(drop=True)
    X, y = train.drop(columns="y"), train["y"].astype(int)
    print(f"\nDROP_DUPLICATES=True: {n_surplus} duplicate rows removed -> {len(train)} rows remain.")
else:
    print("\nDuplicates kept in the training set (DROP_DUPLICATES = False)." if n_surplus else "\nNo exact duplicates found.")

### Cross-validation data

In [ ]:
outer_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)       # final model assessment
prep_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=PREP_SEED)    # preparation experiments

fold_of = np.zeros(len(X), dtype=int)
rows = []
for k, (tr, va) in enumerate(outer_cv.split(X, y), start=1):
    fold_of[va] = k
    rows.append({"fold": k, "n_train": len(tr), "n_validation": len(va),
                 "%y=1 train": round(100 * y.iloc[tr].mean(), 2),
                 "%y=1 validation": round(100 * y.iloc[va].mean(), 2)})
show_table(pd.DataFrame(rows).set_index("fold"), "cv_folds_summary", "Stratified cross-validation folds", decimals=2)
pd.DataFrame({"row": np.arange(len(X)), "fold": fold_of}).to_csv(OUT_DIR / "cv_folds.csv", index=False)

METRICS = ["accuracy", "recall", "precision", "f1"]


def compare(options, base):
    """Cross-validates the reference models under each option; returns (long table, accuracy summary)."""
    rows = []
    for label, kw in options.items():
        cfg = {**base, **kw}
        for name, model in REF_MODELS.items():
            res = cross_validate(build_pipeline(model, **cfg), X, y, cv=prep_cv,
                                 scoring=METRICS, n_jobs=N_JOBS)
            rows.append({"option": label, "model": name, **{m: res[f"test_{m}"].mean() for m in METRICS}})
    long = pd.DataFrame(rows)
    acc = long.pivot(index="option", columns="model", values="accuracy")
    acc["mean_accuracy"] = acc.mean(axis=1)
    return long, acc.loc[list(options)]


def show(long, acc, name, what):
    show_table(acc, f"{name}_accuracy", f"Cross-validated accuracy by {what}", color="best")
    show_table(long.pivot(index="option", columns="model", values=["recall", "precision", "f1"]),
               f"{name}_other_metrics", f"Cross-validated recall, precision and F-measure by {what}")


def pick(acc, override=None):
    """Best option by mean accuracy, but the first option (the simplest) is kept unless another one is better
    by at least MIN_GAIN: differences smaller than that are noise."""
    if override is not None:
        return override
    first, best = acc.index[0], acc["mean_accuracy"].idxmax()
    return best if acc.loc[best, "mean_accuracy"] - acc.loc[first, "mean_accuracy"] >= MIN_GAIN else first


BASE = dict(missing="median", outliers="keep", balancing="none", corr_threshold=None, use_d=True)

### Missing values

In [ ]:
incomplete = X.isna().any(axis=1)
print(f"Rows with at least one missing value: {incomplete.sum()} ({100 * incomplete.mean():.2f}% of training data)")

if incomplete.any():
    ref = REF_MODELS["RandomForest"]
    res = {"drop incomplete rows (complete val. rows)": [],
           "median imputation (same complete val. rows)": [],
           "median imputation (incomplete val. rows only)": []}
    for tr, va in prep_cv.split(X, y):
        Xtr, ytr, Xva, yva = X.iloc[tr], y.iloc[tr], X.iloc[va], y.iloc[va]
        ok_tr, ok_va = ~Xtr.isna().any(axis=1), ~Xva.isna().any(axis=1)
        p = build_pipeline(ref).fit(Xtr[ok_tr], ytr[ok_tr])
        res["drop incomplete rows (complete val. rows)"].append(accuracy_score(yva[ok_va], p.predict(Xva[ok_va])))
        p = build_pipeline(ref).fit(Xtr, ytr)
        pred = pd.Series(p.predict(Xva), index=Xva.index)
        res["median imputation (same complete val. rows)"].append(accuracy_score(yva[ok_va], pred[ok_va]))
        if (~ok_va).any():
            res["median imputation (incomplete val. rows only)"].append(accuracy_score(yva[~ok_va], pred[~ok_va]))
    show_table(pd.Series({k: np.mean(v) for k, v in res.items()}, name="mean_accuracy_RandomForest"),
               "missing_drop_vs_impute", "Dropping incomplete rows vs. imputing (Random Forest, cross-validated accuracy)")

    cmp_missing, acc_missing = compare({"median": {"missing": "median"},
                                        "mean": {"missing": "mean"},
                                        "knn": {"missing": "knn"}}, BASE)
    show(cmp_missing, acc_missing, "missing_methods", "missing-value method")
    best_missing = pick(acc_missing)
else:
    print("No missing values in the training data: simple median imputation stays in the pipeline as a safeguard "
          "for the test set.")
    best_missing = "median"
print("-> missing-value strategy:", best_missing)

### Outliers

In [ ]:
cmp_out, acc_out = compare({"keep": {"outliers": "keep"},
                            "clip": {"outliers": "clip"},
                            "robust": {"outliers": "robust"},
                            "power": {"outliers": "power"}}, {**BASE, "missing": best_missing})
show(cmp_out, acc_out, "outlier_treatments", "outlier treatment")
best_outliers = pick(acc_out)
print("-> outlier treatment:", best_outliers)

### Unbalanced data

In [ ]:
print(y.value_counts(normalize=True).sort_index().round(4).to_string())
cmp_bal, acc_bal = compare({"none": {"balancing": "none"},
                            "smote": {"balancing": "smote"},
                            "undersample": {"balancing": "undersample"}},
                           {**BASE, "missing": best_missing, "outliers": best_outliers})
show(cmp_bal, acc_bal, "balancing_methods", "balancing method")
best_balancing = pick(acc_bal)
print("-> balancing:", best_balancing)

### Feature redundancy

In [ ]:
x_imputed = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(X[num_cols]), columns=num_cols)
pairs = high_corr_pairs(x_imputed, CORR_THRESHOLD)
if len(pairs):
    show_table(pairs, "redundant_pairs", f"Pairs of numeric variables with absolute correlation >= {CORR_THRESHOLD}",
               index=False, decimals=3)
else:
    print(f"No pairs of numeric variables with absolute correlation >= {CORR_THRESHOLD}.")
dropped = [c for k, c in enumerate(num_cols) if k not in CorrelationFilter(CORR_THRESHOLD).fit(x_imputed.values).keep_]
print("Variables dropped by the filter on the full training data:", dropped or "none")

# near-constant variables (low variability)
near_const = X.apply(lambda s: s.value_counts(normalize=True, dropna=False).iloc[0]).sort_values(ascending=False)
show_table(near_const.head(5).rename("share_most_frequent_value"), "near_constant",
           "Variables with the highest share of a single value (low variability)")

cmp_corr, acc_corr = compare({"no filter": {"corr_threshold": None},
                              "0.95": {"corr_threshold": 0.95},
                              "0.90": {"corr_threshold": 0.90},
                              "0.80": {"corr_threshold": 0.80}},
                             {"missing": best_missing, "outliers": best_outliers, "balancing": best_balancing})
show(cmp_corr, acc_corr, "redundancy_thresholds", "correlation threshold")
best_corr = {"no filter": None, "0.95": 0.95, "0.90": 0.90, "0.80": 0.80}[pick(acc_corr)]
print("-> correlation threshold:", best_corr)

# Are the discrete variables redundant with the continuous ones? Predict each d* from the x* alone.
x_only = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(X[x_vars]), columns=x_vars)
d_pred = []
for c in cat_cols:
    acc_c = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=N_JOBS),
                            x_only, X[c], cv=3, scoring="accuracy").mean()
    d_pred.append({"variable": c, "accuracy_from_x": acc_c,
                   "majority_class_accuracy": X[c].value_counts(normalize=True).iloc[0]})
show_table(pd.DataFrame(d_pred).set_index("variable"), "d_predictable_from_x",
           "How well each discrete variable can be predicted from the continuous variables alone (3-fold accuracy)")

cmp_d, acc_d_cmp = compare({"drop d variables": {"use_d": False}, "keep d variables": {"use_d": True}},
                           {"missing": best_missing, "outliers": best_outliers, "balancing": best_balancing,
                            "corr_threshold": best_corr})
show(cmp_d, acc_d_cmp, "discrete_variables", "use of the discrete variables")
best_use_d = {"keep d variables": True, "drop d variables": False}[pick(acc_d_cmp)]
print("-> use the discrete variables:", best_use_d)

### Final preparation configuration

In [ ]:
# Set a value to override the automatic choice (e.g. OVERRIDE["outliers"] = "clip")
OVERRIDE = {"missing": None, "outliers": None, "balancing": None, "corr_threshold": "auto", "use_d": "auto"}

PREP = dict(
    missing=OVERRIDE["missing"] or best_missing,
    outliers=OVERRIDE["outliers"] or best_outliers,
    balancing=OVERRIDE["balancing"] or best_balancing,
    corr_threshold=best_corr if OVERRIDE["corr_threshold"] == "auto" else OVERRIDE["corr_threshold"],
    use_d=best_use_d if OVERRIDE["use_d"] == "auto" else OVERRIDE["use_d"],
)
print("Preparation used for every model:", PREP)
show_table(pd.Series({k: str(v) for k, v in PREP.items()}, name="choice"), "preparation_choices",
           "Preparation choices used for every model")

Xt = make_preprocessor(PREP["missing"], PREP["outliers"], PREP["corr_threshold"], PREP["use_d"]).fit_transform(X, y)
print(f"Design matrix delivered to the models: {Xt.shape[0]} rows x {Xt.shape[1]} columns "
      f"(from {X.shape[1]} original variables)")

## Modeling

In [ ]:
MODELS = {
    "Baseline (majority class)": (DummyClassifier(strategy="most_frequent"), {}),
    "kNN": (KNeighborsClassifier(),
            {"clf__n_neighbors": [3, 5, 7, 11, 15, 21, 31], "clf__weights": ["uniform", "distance"]}),
    "Decision Tree": (DecisionTreeClassifier(random_state=SEED),
                      {"clf__max_depth": [3, 5, 8, 12, None], "clf__min_samples_leaf": [1, 5, 20]}),
    "Naive Bayes": (GaussianNB(),
                    {"clf__var_smoothing": [1e-9, 1e-7, 1e-5, 1e-3, 1e-2, 1e-1, 1.0]}),
    "MLP": (MLPClassifier(max_iter=500, early_stopping=True, random_state=SEED),
            {"clf__hidden_layer_sizes": [(32,), (64,), (64, 32), (128, 64)], "clf__alpha": [1e-5, 1e-4, 1e-2]}),
    "Random Forest": (RandomForestClassifier(random_state=SEED),
                      {"clf__n_estimators": [200, 400], "clf__max_depth": [None, 10, 20],
                       "clf__min_samples_leaf": [1, 3]}),
    "Bagging (trees)": (BaggingClassifier(estimator=DecisionTreeClassifier(random_state=SEED), random_state=SEED),
                        {"clf__n_estimators": [50, 150], "clf__max_samples": [0.7, 1.0]}),
    "AdaBoost": (AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1, random_state=SEED), random_state=SEED),
                 {"clf__n_estimators": [100, 300], "clf__learning_rate": [0.1, 1.0],
                  "clf__estimator__max_depth": [1, 2, 3, 5]}),
    "Gradient Boosting": (GradientBoostingClassifier(random_state=SEED),
                          {"clf__n_estimators": [100, 300], "clf__learning_rate": [0.05, 0.1],
                           "clf__max_depth": [3, 5, 8]}),
}


def metrics_of(y_true, y_pred):
    """Metrics for the positive class (y = 1, discount)."""
    return {"accuracy": accuracy_score(y_true, y_pred),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0)}


def evaluate_model(estimator, grid):
    """Nested CV: returns per-fold metrics, out-of-fold predictions and the selected hyper-parameters."""
    fold_rows, best_params = [], []
    oof = pd.Series(index=X.index, dtype=int)
    inner_cv = StratifiedKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=SEED)
    for k, (tr, va) in enumerate(outer_cv.split(X, y), start=1):
        pipe = build_pipeline(estimator, **PREP)
        if TUNE and grid:
            model = GridSearchCV(pipe, grid, cv=inner_cv, scoring="accuracy", n_jobs=N_JOBS)
            model.fit(X.iloc[tr], y.iloc[tr])
            best_params.append({p.replace("clf__", ""): v for p, v in model.best_params_.items()})
        else:
            model = pipe.fit(X.iloc[tr], y.iloc[tr])
        pred = model.predict(X.iloc[va])
        oof.iloc[va] = pred
        fold_rows.append({"fold": k, **metrics_of(y.iloc[va], pred)})
    return pd.DataFrame(fold_rows).set_index("fold"), oof, best_params

In [ ]:
import time

fold_scores, oof_preds, tuned_params = {}, {}, {}
for name, (estimator, grid) in MODELS.items():
    t0 = time.time()
    fold_scores[name], oof_preds[name], tuned_params[name] = evaluate_model(estimator, grid)
    print(f"{name:<28} accuracy = {fold_scores[name]['accuracy'].mean():.4f} "
          f"(+/- {fold_scores[name]['accuracy'].std():.4f})   [{time.time() - t0:5.0f}s]")

## Results

In [ ]:
def summarise(fold_scores):
    mean = pd.DataFrame({m: s[METRICS].mean() for m, s in fold_scores.items()}).T
    std = pd.DataFrame({m: s[METRICS].std() for m, s in fold_scores.items()}).T
    mean = mean.sort_values("accuracy", ascending=False)
    std = std.loc[mean.index]
    nice = {"accuracy": "Accuracy", "recall": "Recall", "precision": "Precision", "f1": "F-measure"}
    numeric = mean.rename(columns=nice)[["Accuracy", "Recall", "Precision", "F-measure"]]
    formatted = numeric.copy().astype(object)
    for c, c_raw in zip(numeric.columns, ["accuracy", "recall", "precision", "f1"]):
        formatted[c] = [f"{m:.4f} ± {s:.4f}" for m, s in zip(mean[c_raw], std[c_raw])]
    numeric.index.name = formatted.index.name = "Model"
    return numeric, formatted


results_numeric, results_fmt = summarise(fold_scores)
show_table(results_fmt, "model_comparison", "Model comparison: mean ± std over the outer folds (positive class: y = 1)",
           highlight_row=next(m for m in results_numeric.index if not m.startswith("Baseline")))
results_numeric.round(4).to_csv(TABLE_DIR / "model_comparison_numeric.csv")
# accuracy, recall, precision and F-measure of every model in every outer fold (for plots and the report)
pd.concat(fold_scores, names=["model", "fold"]).reset_index().to_csv(TABLE_DIR / "fold_scores.csv", index=False)

In [ ]:
# Accuracy per outer fold, and all metrics side by side
order = results_numeric.index.tolist()
acc_folds = pd.DataFrame({m: fold_scores[m]["accuracy"] for m in order})

fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [1.1, 1]})
sns.boxplot(data=acc_folds, orient="h", ax=axes[0], color=C_BOX)
sns.stripplot(data=acc_folds, orient="h", ax=axes[0], color="black", size=4)
axes[0].axvline(y.value_counts(normalize=True).max(), color="red", ls="--", lw=1, label="majority-class baseline")
axes[0].set(xlabel="accuracy per outer fold", title="Accuracy across folds")
axes[0].legend(loc="lower right")

results_numeric.plot(kind="barh", ax=axes[1], width=0.85, color=METRIC_COLORS)
axes[1].invert_yaxis()
axes[1].set(xlim=(0, 1), title="Mean metrics by model", ylabel="")
axes[1].legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=4, fontsize=9, frameon=False)
fig.tight_layout()
savefig("model_comparison")
plt.show()

In [ ]:
# Best model (excluding the baseline): confusion matrix from the out-of-fold predictions and tuned hyper-parameters
best_name = [m for m in results_numeric.index if not m.startswith("Baseline")][0]
print("Best model:", best_name)

cm = confusion_matrix(y, oof_preds[best_name])
share = cm / cm.sum(axis=1, keepdims=True)           # share of each true class
good, bad = np.array(to_rgb(BLUE)), np.array(to_rgb(ORANGE))
img = np.array([[1 - share[i, j] * (1 - (good if i == j else bad)) for j in range(2)] for i in range(2)])
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(img)                                        # blue = correct predictions, orange = errors
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]}\n({100 * share[i, j]:.1f}%)", ha="center", va="center", fontsize=11,
                color="white" if share[i, j] > 0.6 else "#222222")
ax.set_xticks([0, 1]); ax.set_xticklabels(["no discount", "discount"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["no discount", "discount"])
ax.set(xlabel="Predicted label", ylabel="True label")
ax.grid(False)
ax.set_title(f"{best_name} — out-of-fold confusion matrix")
savefig("confusion_matrix_best")
pd.DataFrame(confusion_matrix(y, oof_preds[best_name]), index=["true_no_discount", "true_discount"],
             columns=["pred_no_discount", "pred_discount"]).to_csv(TABLE_DIR / "confusion_matrix_best.csv")
plt.show()

if TUNE:
    hp = pd.DataFrame({m: [", ".join(f"{k}={v}" for k, v in p.items()) for p in tuned_params[m]]
                       for m in order if tuned_params[m]}).T
    hp.columns = [f"fold {k}" for k in range(1, N_SPLITS + 1)]
    pd.set_option("display.max_colwidth", 120)
    show_table(hp, "tuned_hyperparameters", "Hyper-parameters selected in each outer fold")

### Final model and submission

In [ ]:
est, grid = MODELS[best_name]
final_pipe = build_pipeline(est, **PREP)
final_model = GridSearchCV(final_pipe, grid, cv=StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED),
                           scoring="accuracy", n_jobs=N_JOBS).fit(X, y) if (TUNE and grid) else final_pipe.fit(X, y)
if TUNE and grid:
    print("Selected hyper-parameters:", final_model.best_params_, f"(CV accuracy {final_model.best_score_:.4f})")

if TEST_FILE.exists():
    test = pd.read_csv(TEST_FILE)
    submission = pd.DataFrame({"i": test["i"], "y": final_model.predict(test[X.columns]).astype(int)})
    submission.to_csv(OUT_DIR / "submission.csv", index=False)
    assert list(submission.columns) == ["i", "y"] and submission["i"].is_unique
    assert set(submission["y"]) <= {0, 1} and len(submission) == len(test)
    print(f"submission.csv written: {len(submission)} rows; predicted share of y=1 = {submission['y'].mean():.3f}"
          f" (training: {y.mean():.3f})")
    display(submission.head())
else:
    print(f"{TEST_FILE} not found: submission not created.")

In [ ]:
# Which variables drive the best model? (permutation importance on an untouched 20% hold-out)
Xa, Xb, ya, yb = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
best_params = final_model.best_params_ if (TUNE and grid) else {}
probe = build_pipeline(est, **PREP).set_params(**best_params).fit(Xa, ya)
imp = permutation_importance(probe, Xb, yb, scoring="accuracy", n_repeats=10, random_state=SEED, n_jobs=N_JOBS)
imp = pd.Series(imp.importances_mean, index=X.columns).sort_values()
print(f"Hold-out accuracy of the probe model: {accuracy_score(yb, probe.predict(Xb)):.4f}")
show_table(imp.sort_values(ascending=False).rename("accuracy_drop"), "permutation_importance",
           "Permutation importance on the hold-out set (drop in accuracy when the variable is shuffled)",
           color="gradient")

fig, ax = plt.subplots(figsize=(5, 0.28 * len(imp) + 1.5))
imp.plot(kind="barh", ax=ax, color=C_IMP)
ax.set(xlabel="drop in accuracy when the variable is shuffled", title=f"Permutation importance — {best_name}")
savefig("permutation_importance")
plt.show()

## Conclusions

In [ ]:
display(results_fmt)
print(f"Best model: {best_name} — accuracy {results_numeric.loc[best_name, 'Accuracy']:.4f}, "
      f"recall {results_numeric.loc[best_name, 'Recall']:.4f}, "
      f"precision {results_numeric.loc[best_name, 'Precision']:.4f}, "
      f"F-measure {results_numeric.loc[best_name, 'F-measure']:.4f}")